# 🚀 Optimize Edilmiş Deprem Yol Hasar Tespiti Model Eğitimi

Bu notebook, önceki eğitimdeki sorunları düzelterek **daha iyi performans** sağlayan bir model eğitir.

### İyileştirmeler:
- ✅ **ImageNet normalizasyonu** (encoder buna göre pretrained)
- ✅ **Focal + Dice birleşik loss** (sınıf dengesizliğini çözer)
- ✅ **CosineAnnealing LR scheduler** (daha iyi convergence)
- ✅ **Gelişmiş augmentation** (uydu görüntülerine özel)
- ✅ **IoU metriği** ile validasyon (gerçek performans ölçümü)
- ✅ **KATE-CD + MedBFS** birleştirilmiş veri seti
- ✅ **mit_b4 encoder** (daha güçlü)

**ÖNEMLİ:** Üst menüden `Çalışma Zamanı` → `Çalışma zamanı türünü değiştir` → **T4 GPU** seçin.

## 1. Kütüphaneler

In [ ]:
!pip install -q datasets opencv-python numpy tqdm scikit-learn torch torchvision segmentation-models-pytorch albumentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.7 MB/s eta 0:00:00


## 2. Dizinleri Oluştur

In [ ]:
import os
import numpy as np
import cv2

IMG_DIR = 'data/images/'
MASK_DIR = 'data/masks/'
TILE_IMG_DIR = 'data/tiles/images/'
TILE_MASK_DIR = 'data/tiles/masks/'
MODEL_DIR = 'models/'

for d in [IMG_DIR, MASK_DIR, TILE_IMG_DIR, TILE_MASK_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)
print('✅ Klasörler hazır!')

✅ Klasörler hazır!


## 3. KATE-CD Veri Seti İndirme

In [ ]:
from datasets import load_dataset

print('KATE-CD Veri seti çekiliyor...')
try:
    dataset_kate = load_dataset('CSCRS/kate-cd')
except Exception as e:
    print(f'Hata: {e}. Alternatif deneniyor...')
    dataset_kate = load_dataset('hkayabilisim/kate-cd')

kate_count = 0
for i, veri in enumerate(dataset_kate['train']):
    post_img = veri.get('post_image') or veri.get('image')
    label_img = veri.get('label') or veri.get('mask')

    if post_img and label_img:
        img_np = np.array(post_img)
        mask_np = np.array(label_img)

        if len(mask_np.shape) > 2:
            mask_np = cv2.cvtColor(mask_np, cv2.COLOR_RGB2GRAY)
        _, mask_np = cv2.threshold(mask_np, 0, 255, cv2.THRESH_BINARY)

        if len(img_np.shape) == 3:
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

        if img_np.shape[:2] == mask_np.shape[:2]:
            cv2.imwrite(os.path.join(IMG_DIR, f'katecd_{i}.png'), img_np)
            cv2.imwrite(os.path.join(MASK_DIR, f'katecd_{i}.png'), mask_np)
            kate_count += 1

print(f'✅ KATE-CD: {kate_count} görüntü kaydedildi.')

KATE-CD Veri seti çekiliyor...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/371M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/40.5M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/34.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/404 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/44 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/38 [00:00<?, ? examples/s]

✅ KATE-CD: 404 görüntü kaydedildi.


## 4. MedBFS Veri Seti İndirme

In [ ]:
import urllib.request
import zipfile
from tqdm import tqdm

dataset_url = 'https://zenodo.org/records/11647956/files/Turkey.zip?download=1'
zip_path = '/content/Turkey.zip'
extract_path = '/content/MEDBFS_Turkey'

if not os.path.exists(zip_path):
    print('MedBFS İndiriliyor... (~3.3GB)')
    urllib.request.urlretrieve(dataset_url, zip_path)
    print('İndirme tamamlandı.')

if not os.path.exists(extract_path):
    print('MedBFS Çıkarılıyor...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print('Çıkarma tamamlandı.')

medbfs_img_dir = '/content/MEDBFS_Turkey/Turkey/Image/Post-disaster'
medbfs_lbl_dir = '/content/MEDBFS_Turkey/Turkey/Label'

medbfs_count = 0
if os.path.exists(medbfs_img_dir) and os.path.exists(medbfs_lbl_dir):
    for img_name in tqdm(os.listdir(medbfs_img_dir)):
        if not img_name.endswith('.tif'):
            continue
        src_img = os.path.join(medbfs_img_dir, img_name)
        src_lbl = os.path.join(medbfs_lbl_dir, img_name)
        if not os.path.exists(src_lbl):
            continue
        img = cv2.imread(src_img)
        mask = cv2.imread(src_lbl, cv2.IMREAD_GRAYSCALE)
        if img is None or mask is None:
            continue
        mask = np.where(mask > 0, 255, 0).astype(np.uint8)
        base = os.path.splitext(img_name)[0]
        if img.shape[:2] == mask.shape[:2]:
            cv2.imwrite(os.path.join(IMG_DIR, f'medbfs_{base}.png'), img)
            cv2.imwrite(os.path.join(MASK_DIR, f'medbfs_{base}.png'), mask)
            medbfs_count += 1

print(f'✅ MedBFS: {medbfs_count} görüntü eklendi.')
print(f'📊 Toplam veri: {kate_count + medbfs_count} görüntü')

MedBFS İndiriliyor... (~3.3GB)
İndirme tamamlandı.
MedBFS Çıkarılıyor...
Çıkarma tamamlandı.


100%|██████████| 438/438 [00:09<00:00, 46.62it/s]

✅ MedBFS: 438 görüntü eklendi.
📊 Toplam veri: 842 görüntü


## 5. Tiling (Parçalama)
Görüntüleri 512×512 parçalara böl. Negatif (hasarsız) örneklerin %95'i atılarak sınıf dengesizliği azaltılıyor.

In [ ]:
tile_size = 512
# 256 stride ile overlap tiling — daha fazla eğitim verisi üretir
tile_stride = 256
print(f'🔪 Tiling başladı (size={tile_size}, stride={tile_stride})...')

positive_count = 0
negative_count = 0

for img_name in tqdm(os.listdir(IMG_DIR)):
    img_path = os.path.join(IMG_DIR, img_name)
    mask_path = os.path.join(MASK_DIR, img_name)
    if not os.path.exists(mask_path):
        continue
    img = cv2.imread(img_path)
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if img is None or mask is None:
        continue
    if img.shape[:2] != mask.shape[:2]:
        continue
    h, w = img.shape[:2]

    for y in range(0, h - tile_size + 1, tile_stride):
        for x in range(0, w - tile_size + 1, tile_stride):
            img_tile = img[y:y+tile_size, x:x+tile_size]
            mask_tile = mask[y:y+tile_size, x:x+tile_size]

            if img_tile.shape[0] != tile_size or img_tile.shape[1] != tile_size:
                continue

            has_damage = np.sum(mask_tile) > 0

            # Hasarsız tile'ların %95'ini atla (dengesizliği önle)
            if not has_damage and np.random.rand() > 0.05:
                continue

            base = os.path.splitext(img_name)[0]
            tile_name = f'{base}_y{y}_x{x}.png'
            cv2.imwrite(os.path.join(TILE_IMG_DIR, tile_name), img_tile)
            cv2.imwrite(os.path.join(TILE_MASK_DIR, tile_name), mask_tile)

            if has_damage:
                positive_count += 1
            else:
                negative_count += 1

total = positive_count + negative_count
print(f'✅ Tiling tamamlandı! Toplam: {total} tile')
print(f'   Hasarlı: {positive_count} ({100*positive_count/max(total,1):.1f}%)')
print(f'   Hasarsız: {negative_count} ({100*negative_count/max(total,1):.1f}%)')

🔪 Tiling başladı (size=512, stride=256)...


100%|██████████| 842/842 [00:23<00:00, 36.59it/s]

✅ Tiling tamamlandı! Toplam: 842 tile
   Hasarlı: 842 (100.0%)
   Hasarsız: 0 (0.0%)


## 6. Model Eğitimi (Optimize Edilmiş)
- **Encoder:** mit_b4 (ImageNet pretrained)
- **Loss:** Focal + Dice (0.5 + 0.5)
- **Optimizer:** AdamW + CosineAnnealingWarmRestarts
- **Normalizasyon:** ImageNet mean/std
- **Augmentation:** Uydu görüntülerine özel gelişmiş pipeline
- **Metrik:** IoU (Intersection over Union)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp
import torch.optim as optim

# ────────────────────────────────────────
# ImageNet normalizasyon değerleri
# ────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

# ────────────────────────────────────────
# Dataset sınıfı (ImageNet normalizasyonlu)
# ────────────────────────────────────────
class DepremDataset(Dataset):
    def __init__(self, file_list, img_dir, mask_dir, transform=None):
        self.file_list = file_list
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        img_name = self.file_list[idx]
        image = cv2.imread(os.path.join(self.img_dir, img_name))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(os.path.join(self.mask_dir, img_name), 0)
        mask = np.where(mask > 0, 1.0, 0.0).astype(np.float32)

        if self.transform:
            aug = self.transform(image=image, mask=mask)
            image = aug['image']  # ToTensorV2 çıkışı: [C, H, W] float tensor
            mask = aug['mask']
        else:
            # Fallback: elle normalize et
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            for c in range(3):
                image[c] = (image[c] - IMAGENET_MEAN[c]) / IMAGENET_STD[c]
            mask = torch.from_numpy(mask)

        return image, mask.long()

# ────────────────────────────────────────
# Augmentation pipeline
# ────────────────────────────────────────
train_transform = A.Compose([
    # Geometrik
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=20, p=0.5,
                       border_mode=cv2.BORDER_REFLECT_101),
    A.RandomResizedCrop(size=(512, 512), scale=(0.7, 1.0), ratio=(0.9, 1.1), p=0.3),

    # Renk / Parlaklık
    A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.05, p=0.5),
    A.CLAHE(clip_limit=4.0, tile_grid_size=(8, 8), p=0.2),

    # Gürültü / Blur
    A.GaussianBlur(blur_limit=(3, 5), p=0.15),
    A.GaussNoise(std_range=(0.02, 0.1), p=0.15),

    # ImageNet normalizasyon + Tensor dönüşümü
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

val_transform = A.Compose([
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2(),
])

# ────────────────────────────────────────
# IoU metriği
# ────────────────────────────────────────
def compute_iou(preds, targets, threshold=0.5):
    """Binary IoU hesapla."""
    preds_bin = (torch.sigmoid(preds) > threshold).float()
    targets_f = targets.float()
    intersection = (preds_bin * targets_f).sum()
    union = preds_bin.sum() + targets_f.sum() - intersection
    if union == 0:
        return 1.0  # Her ikisi de boş
    return (intersection / union).item()

# ────────────────────────────────────────
# Veri yükleme
# ────────────────────────────────────────
save_path = 'models/optimized_mitb4_focal_dice.pth'

all_tiles = sorted(os.listdir(TILE_IMG_DIR))
print(f'Toplam tile sayısı: {len(all_tiles)}')

if len(all_tiles) == 0:
    print('HATA: Eğitim verisi bulunamadı!')
else:
    train_files, val_files = train_test_split(all_tiles, test_size=0.15, random_state=42)
    print(f'Train: {len(train_files)}, Val: {len(val_files)}')

    train_dataset = DepremDataset(train_files, TILE_IMG_DIR, TILE_MASK_DIR, transform=train_transform)
    val_dataset = DepremDataset(val_files, TILE_IMG_DIR, TILE_MASK_DIR, transform=val_transform)

    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory=True)

    # ────────────────────────────────────────
    # Model (mit_b4 — daha güçlü encoder)
    # ────────────────────────────────────────
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f'Cihaz: {device}')

    model = smp.Segformer(
        encoder_name='mit_b4',
        encoder_weights='imagenet',
        in_channels=3,
        classes=1
    ).to(device)

    # ────────────────────────────────────────
    # Combined Loss: Focal + Dice
    # ────────────────────────────────────────
    dice_loss_fn = smp.losses.DiceLoss(mode='binary')
    focal_loss_fn = smp.losses.FocalLoss(mode='binary', alpha=0.25, gamma=2.0)

    def combined_loss(pred, target):
        return 0.5 * dice_loss_fn(pred, target) + 0.5 * focal_loss_fn(pred, target)

    # ────────────────────────────────────────
    # Optimizer + Scheduler
    # ────────────────────────────────────────
    optimizer = optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2, eta_min=1e-6)

    # ────────────────────────────────────────
    # Eğitim döngüsü
    # ────────────────────────────────────────
    epochs = 50
    patience = 8
    best_val_iou = 0.0
    best_val_loss = float('inf')
    epochs_no_improve = 0

    print(f'🔥 EĞİTİM BAŞLIYOR! (Max {epochs} epoch, Patience: {patience})')
    print(f'   Model: Segformer mit_b4 | Loss: Focal+Dice | LR: CosineAnnealing')
    print('=' * 70)

    for epoch in range(epochs):
        # ── Train ──
        model.train()
        train_loss = 0
        train_iou = 0
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')

        for images, masks in pbar:
            images = images.to(device)
            masks = masks.to(device).unsqueeze(1).float()

            optimizer.zero_grad()
            outputs = model(images)
            loss = combined_loss(outputs, masks)
            loss.backward()

            # Gradient clipping (kararsız eğitimi önler)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()

            train_loss += loss.item()
            train_iou += compute_iou(outputs, masks)
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})

        scheduler.step()

        # ── Validation ──
        model.eval()
        val_loss = 0
        val_iou = 0
        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device)
                masks = masks.to(device).unsqueeze(1).float()
                outputs = model(images)
                loss = combined_loss(outputs, masks)
                val_loss += loss.item()
                val_iou += compute_iou(outputs, masks)

        avg_train_loss = train_loss / len(train_loader)
        avg_val_loss = val_loss / len(val_loader)
        avg_train_iou = train_iou / len(train_loader)
        avg_val_iou = val_iou / len(val_loader)
        current_lr = optimizer.param_groups[0]['lr']

        print(f'👉 Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | '
              f'Train IoU: {avg_train_iou:.4f} | Val IoU: {avg_val_iou:.4f} | LR: {current_lr:.6f}')

        # IoU bazlı model kaydetme (loss yerine IoU daha güvenilir)
        if avg_val_iou > best_val_iou:
            best_val_iou = avg_val_iou
            best_val_loss = avg_val_loss
            epochs_no_improve = 0
            torch.save(model.state_dict(), save_path)
            print(f'✨ Yeni en iyi model! Val IoU: {best_val_iou:.4f} → {save_path}')
        else:
            epochs_no_improve += 1
            print(f'⚠️ Gelişme yok. Patience: {epochs_no_improve}/{patience}')
            if epochs_no_improve >= patience:
                print(f'⏳ Early Stopping! En iyi Val IoU: {best_val_iou:.4f}')
                break

    print('=' * 70)
    print(f'🏆 Eğitim tamamlandı! En iyi Val IoU: {best_val_iou:.4f} | Val Loss: {best_val_loss:.4f}')

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


Toplam tile sayısı: 842
Train: 715, Val: 127
Cihaz: cuda


config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/246M [00:00<?, ?B/s]

🔥 EĞİTİM BAŞLIYOR! (Max 50 epoch, Patience: 8)
   Model: Segformer mit_b4 | Loss: Focal+Dice | LR: CosineAnnealing


...

👉 Epoch 21 | Train Loss: 0.2351 | Val Loss: 0.2541 | Train IoU: 0.4092 | Val IoU: 0.3747 | LR: 0.000159
⚠️ Gelişme yok. Patience: 6/8


Epoch 22/50 [Train]: 100%|██████████| 90/90 [03:11<00:00,  2.13s/it, loss=0.2904]


👉 Epoch 22 | Train Loss: 0.2247 | Val Loss: 0.2325 | Train IoU: 0.4273 | Val IoU: 0.4162 | LR: 0.000146
⚠️ Gelişme yok. Patience: 7/8


Epoch 23/50 [Train]: 100%|██████████| 90/90 [03:11<00:00,  2.13s/it, loss=0.2710]


👉 Epoch 23 | Train Loss: 0.2229 | Val Loss: 0.2458 | Train IoU: 0.4333 | Val IoU: 0.3941 | LR: 0.000131
⚠️ Gelişme yok. Patience: 8/8
⏳ Early Stopping! En iyi Val IoU: 0.4276
🏆 Eğitim tamamlandı! En iyi Val IoU: 0.4276 | Val Loss: 0.2287


## 7. Modeli İndir
En iyi model ağırlıklarını bilgisayara indir.

**NOT:** İndirilen `.pth` dosyasını `deprem_yol_projesi/models/` klasörüne koy ve `app.py` sidebar'ından dosya yolunu güncelle.

In [ ]:
from google.colab import files

if os.path.exists(save_path):
    print(f'Model boyutu: {os.path.getsize(save_path) / 1e6:.1f} MB')
    files.download(save_path)
else:
    print('Model dosyası bulunamadı!')

Model boyutu: 245.8 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>